In [13]:
import torch, ultralytics
import numpy as np
import cv2
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "no GPU - Runtime > Change runtime type > GPU"

torch 2.7.1+cu118 | cuda True
ultralytics 8.4.139


In [14]:
# --- data -------------------------------------------------------------------
# Leave DRIVE_VIDEO_DIR empty to search your Drive for the videos by name -- a
# share link gives a folder id, not a path, so the path cannot be derived from it.
# Set it explicitly only if the search picks the wrong copy.
DRIVE_VIDEO_DIR = ""
VIDEO_GLOB = "29-gameplay-*.mp4"
USE_DRIVE = True          # False -> browser upload widget instead

# --- training ---------------------------------------------------------------
N_TRAIN, N_VAL = 4000, 600   # synthetic scenes
EPOCHS = 60
BATCH = 32                   # drop to 16 if the T4 runs out of memory
IMGSZ = 512                  # must stay equal to CANON_SIZE

# --- reconstruction ---------------------------------------------------------
SAMPLE_FPS = 4.0          # how often to look at the video
CONF = 0.30               # detector threshold; voting tolerates a permissive value

# --- rules ------------------------------------------------------------------
# Who led the very first trick. The video cannot tell you this; in 29 it is the
# bid winner. Every later leader is derived (winner of a trick leads the next).
FIRST_LEADER = 1
DEFAULT_TRUMP_CODE = "S"  # S/H/D/C. Trump is not read off the table yet.
BID_TEAM, BID = "A", 16   # team A = players 1 & 3, team B = 2 & 4

In [15]:
VIDEO1_PATH="Z:\\coding\\PR project\\29-card-game\\29-gameplay-1 - Converted.mp4"
VIDEO2_PATH="Z:\\coding\\PR project\\29-card-game\\29-gameplay-2.mp4"
TRIMMED_PATH="Z:\\coding\\PR project\\29-card-game\\trimmed.mp4"

In [21]:
from pathlib import Path
# just sorts the video according to the path
VIDEOS = sorted(str(path) for path in Path("29-card-game").glob("*.mp4"))
TRAIN_DIR="Z:\\coding\\PR project\\dataset\\train"
VAL_DIR="Z:\\coding\\PR project\\dataset\\val"
OUTPUT_DIR="Z:\\coding\\PR project\\dataset"
IMAGES_DIR=Path("Z:\\coding\\PR project\\dataset\\29-Card-Detection-Dataset-Annotated\\images")
LABELS_DIR=Path("Z:\\coding\\PR project\\dataset\\29-Card-Detection-Dataset-Annotated\\labels")

### Training attempt - 01 (Dont Comment out and run the cell if Video -> Image conversion has already been done )

* Trying to convert the video into images
* taking a sample of 4 fps and saving all it in \train folder
* then annotation needs to be done
* Working on only the first video

In [ ]:
# sample_fps = 4.0
# saved = { "train": 0, "val": 0}

# cap = cv2.VideoCapture(VIDEO1_PATH)

# video_fps = cap.get(cv2.CAP_PROP_FPS)
# video_fps

# frame_step = max(1, round(video_fps / sample_fps))
# print(frame_step)

# frame_index = 0
# saved_from_video = 0
# split = "train" # putting every frame into the training set now before labelling

# while True:
#         ok, frame = cap.read()
#         if not ok:
#             print("Video reading has ended, no more frames can be read")
#             break

#         if frame_index % frame_step == 0: # stops the look when there are no more frames
#             filename = (
#                 f"{Path(VIDEO2_PATH).stem}"
#                 f"_frame_{frame_index:08d}.jpg"
#             ) # naming should be something like '29-gameplay-1_frame_00000120.jpg'
#             output_path =  OUTPUT_DIR + "/" + filename

#             cv2.imwrite(str(output_path), frame)
#             saved[split] += 1
#             saved_from_video += 1

#         frame_index += 1

# cap.release()
# print(f"{Path(VIDEO1_PATH).name}: {saved_from_video} frames -> {split}")

# print(f"Training images:   {saved['train']}")
# print(f"Validation images: {saved['val']}")

8
Video reading has ended, no more frames can be read
29-gameplay-1 - Converted.mp4: 1546 frames -> train
Training images:   1546
Validation images: 0


In [ ]:
# CANON_SIZE = 512
# CANON_RADIUS = 230.0
# CANON_CENTER = (CANON_SIZE / 2.0, CANON_SIZE / 2.0)

# def _table_disc_mask():
#     """The whole table top, not just the play area.

#     Emptiness is scored over the entire table on purpose. Any real face-up card
#     left in a background is an *unlabelled* card in a training image, which teaches
#     the detector to ignore exactly what it is meant to find -- and that applies
#     wherever on the table it sits, not only in the middle. The face-down hand piles
#     are always present and so contribute a near-constant offset that ranking
#     absorbs harmlessly.
#     """
#     yy, xx = np.mgrid[0:CANON_SIZE, 0:CANON_SIZE]
#     return np.hypot(xx - CANON_CENTER[0], yy - CANON_CENTER[1]) < 0.98 * CANON_RADIUS

# def harvest_backgrounds(video_paths, tracker_cls, max_per_video=60, sample_every=2.0):
#     """Collect canonical table crops whose play area is as empty as possible.

#     Emptiness is *ranked*, not thresholded. The cane weave's own highlights read as
#     bright and desaturated, so even a bare table scores ~8% "card-like" pixels in
#     the table -- any absolute cutoff either takes everything or nothing. Taking
#     the lowest-scoring frames per video needs no tuned constant and adapts to
#     whatever the lighting happens to be.

#     Two passes, so only the selected frames are ever held in memory: score first,
#     then re-read the winners using the table fit recorded alongside each score.
#     """
#     play = _table_disc_mask()
#     backgrounds = []

#     for path in video_paths:
#         cap = cv2.VideoCapture(path)
#         print(cap)
#         fps = max(cap.get(cv2.CAP_PROP_FPS), 1.0)
#         step = max(1, int(fps * sample_every))

#         tracker = tracker_cls()
#         scored = []
#         i = -1
#         while True:  # sequential decode; seeking per sample is far slower
#             if not cap.grab():
#                 break
#             i += 1
#             if i % step:
#                 continue
#             ok, frame = cap.retrieve()
#             if not ok:
#                 break
#             fit = tracker.update(frame)
#             if fit is None:
#                 continue
#             hsv = cv2.cvtColor(fit.warp(frame), cv2.COLOR_BGR2HSV)
#             print(hsv)
#             cardish = (hsv[:, :, 2] > 150) & (hsv[:, :, 1] < 60)
#             scored.append((float((cardish & play).sum() / play.sum()), i, fit))

#         # Only the winners are re-read, so peak memory stays at max_per_video frames.
#         scored.sort(key=lambda t: t[0])
#         for _, i, fit in scored[:max_per_video]:
#             cap.set(cv2.CAP_PROP_POS_FRAMES, i)
#             ok, frame = cap.read()
#             if ok:
#                 backgrounds.append(fit.warp(frame))
#         cap.release()

#     return backgrounds

In [ ]:
# from card_faces import load_gallery
# from synthetic_canon import make_scene
# from p29.vision.registration import TableTracker

# # GALLERY = load_gallery("data/reference")  # real scans win when present
# BACKGROUNDS = harvest_backgrounds(VIDEOS, TableTracker, max_per_video=60)

# # print(BACKGROUNDS)
# # print("empty-table backgrounds harvested:", len(BACKGROUNDS))
# # assert BACKGROUNDS, "no clean backgrounds found - check registration above"

< cv2.VideoCapture 000001BB5F4E3D70>
[[[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [126  80  16]
  [126  85  15]
  [126  91  14]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [126  80  16]
  [126  85  15]
  [126  85  15]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [126  80  16]
  [126  85  15]
  [126  85  15]]

 ...

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [156 120  34]
  [156 120  34]
  [156 120  34]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [156 120  34]
  [156 124  33]
  [156 124  33]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [156 120  34]
  [156 124  33]
  [156 124  33]]]
[[[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [124 136  15]
  [124 136  15]
  [128 136  15]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [120 136  15]
  [120 136  15]
  [120 136  15]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [120 136  15]
  [120 136  15]
  [120 136  15]]

 ...

 [[  0   0   0]

### annotation has been done using label-studio, now Have to split the training and validation dataset  before training

### rest are moved to the /train folder

In [23]:
import random
import shutil
from pathlib import Path

# =========================
# Configuration
# =========================

VAL_IMAGES_DIR = IMAGES_DIR / "val"
VAL_LABELS_DIR = LABELS_DIR /  "val"

VAL_COUNT = 40
SEED = 42

# =========================
# Get images
# =========================

image_extensions = {".jpg"} # only jpg images are there

images = [
    f for f in IMAGES_DIR.iterdir()
    if f.is_file() and f.suffix.lower() in image_extensions
]

print(f"Found {len(images)} images.")

if len(images) < VAL_COUNT:
    raise ValueError(
        f"Only {len(images)} images found, "
        f"but requested {VAL_COUNT} validation images."
    )

# =========================
# Shuffle
# =========================

random.seed(SEED)
random.shuffle(images)

# Pick the last 40
val_images = images[-VAL_COUNT:]

print("\nValidation images:")

# =========================
# Move images + labels
# =========================

for image_path in val_images:

    # Corresponding label
    label_path = LABELS_DIR / f"{image_path.stem}.txt"

    if not label_path.exists():
        print(f"WARNING: Label not found for {image_path.name}")
        continue

    # Move image
    shutil.move(
        str(image_path),
        str(VAL_IMAGES_DIR / image_path.name)
    )

    # Move label
    shutil.move(
        str(label_path),
        str(VAL_LABELS_DIR / label_path.name)
    )

    print(f"  {image_path.name}")

print("\nDone!")
print(f"Moved {len(val_images)} images to: {VAL_IMAGES_DIR}")
print(f"Moved corresponding labels to: {VAL_LABELS_DIR}")

Found 211 images.

Validation images:
  dea1dfb9-29-gameplay-1_-_Converted_frame_00012096.jpg
  71e9f354-29-gameplay-1_-_Converted_frame_00007464.jpg
  ee68f128-29-gameplay-2_frame_00003648.jpg
  3287a19f-29-gameplay-2_frame_00001224.jpg
  005d0a7f-29-gameplay-2_frame_00005712.jpg
  5e8151e4-29-gameplay-1_-_Converted_frame_00005576.jpg
  bc8d026b-29-gameplay-1_-_Converted_frame_00003848.jpg
  8ca36a4a-29-gameplay-2_frame_00005576.jpg
  463793a8-29-gameplay-1_-_Converted_frame_00003776.jpg
  875fbc3c-29-gameplay-1_-_Converted_frame_00001296.jpg
  f3d3d42e-29-gameplay-2_frame_00003768.jpg
  e2a8871a-29-gameplay-1_-_Converted_frame_00007376.jpg
  d1e5065d-29-gameplay-1_-_Converted_frame_00003784.jpg
  e5924113-29-gameplay-2_frame_00001936.jpg
  3df321a3-29-gameplay-2_frame_00009744.jpg
  ae9cfca8-29-gameplay-1_-_Converted_frame_00001328.jpg
  f964a666-29-gameplay-2_frame_00003760.jpg
  c2fd1ae6-29-gameplay-2_frame_00007448.jpg
  995a83f3-29-gameplay-2_frame_00000880.jpg
  4df41fdf-29-game

### Finetuning now on top of the pretrained code

In [24]:
from ultralytics import YOLO
import torch

model = YOLO("Z:\coding\PR project\card-detection\pretrained-weights\yolov8m_tuned.pt")


<>:4: SyntaxWarning: invalid escape sequence '\c'
<>:4: SyntaxWarning: invalid escape sequence '\c'
C:\Users\nnahe\AppData\Local\Temp\ipykernel_5288\1509084062.py:4: SyntaxWarning: invalid escape sequence '\c'
  model = YOLO("Z:\coding\PR project\card-detection\pretrained-weights\yolov8m_tuned.pt")


In [ ]:

# print(model.model.model)
detect_layer_index = 22 # the model has 22 layer

for name, param in model.model.named_parameters():
    layer_idx = int(name.split(".")[1]) 
    if layer_idx == detect_layer_index:
        param.requires_grad = True # freeze 22nd layer
    else:
        param.requires_grad = False # freeze others

print(torch.cuda.is_available())

# Fine-tune on your 32-class dataset
model.train(
    data="Z:\coding\PR project\dataset\dataset.yaml",
    # patience=20,  # instead of guessing epochs, it stops training when the loss converges
    epochs=30,
    imgsz=640,
    pretrained=True,
    # device=0 # space issue thus it is trained on cpu
)

True
New https://pypi.org/project/ultralytics/8.4.154 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.139  Python-3.12.7 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Z:\coding\PR project\dataset\dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=

<>:15: SyntaxWarning: invalid escape sequence '\c'
<>:15: SyntaxWarning: invalid escape sequence '\c'
C:\Users\nnahe\AppData\Local\Temp\ipykernel_5288\642563216.py:15: SyntaxWarning: invalid escape sequence '\c'
  data="Z:\coding\PR project\dataset\dataset.yaml",   # defines nc: 32 and your class names


 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  2    517632  ultralytics.nn.modules.block.C2f             [576, 192, 2]                 
 16                  -1  1    332160  ultralytics.nn.modules.conv.Conv             [192, 192, 3, 2]              
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  2   1846272  ultralytics.nn.modules.block.C2f             [576, 384, 2]                 
 19                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  4,  5,  6,  7,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000026048C84530>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,    